# Test neighborhood-level approach:

Note that the method below uses distances in latent space and so is not graph-based anymore

## Prep:

In [ ]:
!pip install nb_black

In [ ]:
!pip install scanpy

load modules:

In [ ]:
%load_ext nb_black

In [ ]:
import numpy as np
import scanpy as sc
import pandas as pd
import random
import os
from os.path import abspath
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scipy.spatial.distance import cdist
import tqdm

import matplotlib.cm
from matplotlib.colors import rgb2hex

set parameters and paths:

In [ ]:
dir_res = abspath(
    os.path.join(os.getcwd(), os.pardir, "Results", "Neighborhood_based_HLCA_weighted")
)

path_hlca_ext = abspath(
    os.path.join(os.getcwd(), os.pardir, "Data", "HLCA_full_tranf.h5ad")
)

In [ ]:

Sheppard_IPF_filtered = f"{dir_res}/Sheppard_2020_Fibroblasts_IPF/All_filter/"
# whole query+reference
Sheppard_healthy = f"{dir_res}/Sheppard_2020_Healthy/"
Sheppard_IPF = f"{dir_res}/Sheppard_2020_IPF/"
Sheppard_ScILD = f"{dir_res}/Sheppard_2020_ScILD/"
# Schiller 2020
Schiller_ELF = f"{dir_res}/Schiller_2020_Endstage_Lung_Fibrosis/"
Schiller_healthy = f"{dir_res}/Schiller_2020_Healthy/"
# Kaminski_2020
Kaminski_healthy = f"{dir_res}/Kaminski_2020_Healthy/"
Kaminski_IPF = f"{dir_res}/Kaminski_2020_IPF/"
Kaminski_COPD = f"{dir_res}/Kaminski_2020_COPD/"


In [ ]:
k_list = [
    500,
    1000,
    2000,
    3000,
    4000,
    5000,
    8000,
    10000,
    12000,
    15000,
    20000,
    30000,
    50000,
]

helperfunctions:

In [ ]:
def load_stats(
    path,
    k_list=[50, 500, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000],
    prop=500,
    runs=20,
):
    list_results = []
    for k in k_list:
        n_list = []
        frac_list = []
        for i in range(0, runs):
            try:
                stats = pd.read_csv(f"{path}pvalues_{k}_{prop}_{i}.csv").dropna()
            except:
                continue
            if len(stats) < 1:
                continue
            n_significant = len(stats[stats.pvalue_corr < 0.05])
            n_list.append(n_significant)
            frac_list.append(n_significant / len(stats))
        if len(n_list) < 1:
            continue
        list_results.append(
            (k, np.mean(n_list), np.std(n_list), np.mean(frac_list), np.std(frac_list))
        )

    return pd.DataFrame(
        data=list_results,
        columns=["k", "n_significant", "std_significant", "fraction", "std_fraction"],
    )

In [ ]:
def load_stats_prop(
    path,
    k_list=[50, 500, 1000, 5000],
    prop_list=[10, 50, 100, 200, 500, 700, 1000, 1500, 2000],
    runs=10,
):
    list_results = []
    for k in k_list:
        n_list = []
        frac_list = []
        for p in prop_list:
            for i in range(0, runs):
                stats = pd.read_csv(f"{path}pvalues_{k}_{p}_{i}.csv").dropna()
                if len(stats) < 1:
                    continue
                n_significant = len(stats[stats.pvalue_corr < 0.05])
                n_list.append(n_significant)
                frac_list.append(n_significant / len(stats))
            if len(n_list) < 1:
                continue
            list_results.append(
                (
                    k,
                    p,
                    np.mean(n_list),
                    np.std(n_list),
                    np.mean(frac_list),
                    np.std(frac_list),
                )
            )

    return pd.DataFrame(
        data=list_results,
        columns=[
            "k",
            "prop",
            "n_significant",
            "std_significant",
            "fraction",
            "std_fraction",
        ],
    )

In [ ]:
def load_pvalues(path, k, p, i):
    try:
        return pd.read_csv(f"{path}pvalues_{k}_{p}_{i}.csv").dropna()
    except:
        print("No such data")
        return None

In [ ]:
def get_aggregation_per_cell_type(adata_ext_h, path_h, path_d, k, prop, runs):
    pvalues_celltype = []
    for r in range(runs):
        pvalues_h = load_pvalues(path_h, k, prop, r)
        pvalues_d = load_pvalues(path_d, k, prop, r)

        if pvalues_h is None and pvalues_d is None:
            print(f"skip run {r}")
            continue
        if not (pvalues_h is None):
            indices_h = pvalues_h["Unnamed: 0"]
            pvalues_h.index = indices_h
            healthy_cts = adata_ext_h.obs.loc[indices_h, "cell_label"]

            for ct in healthy_cts.cat.categories:
                idx = healthy_cts[healthy_cts == ct].index.tolist()
                pvals = pvalues_h.loc[idx, "pvalue_corr"]
                pvalue = scipy.stats.combine_pvalues(pvals).pvalue
                pvalues_celltype.append((r, ct, "healthy", pvalue))

        if not (pvalues_d is None):
            indices_d = pvalues_d["Unnamed: 0"]
            pvalues_d.index = indices_d
            disease_cts = adata_ext_h.obs.loc[indices_d, "cell_label"]
            for ct in disease_cts.cat.categories:
                idx = disease_cts[disease_cts == ct].index.tolist()
                pvals = pvalues_d.loc[idx, "pvalue_corr"]
                pvalue = scipy.stats.combine_pvalues(pvals).pvalue
                pvalues_celltype.append((r, ct, "disease", pvalue))

    pvalues_celltype = pd.DataFrame(
        pvalues_celltype, columns=["run", "cell_label", "condition", "p_value"]
    )

    # print(pvalues_celltype)

    return pvalues_celltype

In [ ]:
def visualize_nhood(
    subset_adata,
    distances,
    center_cell,
    k,
    dataset,
    return_grouped=False,
    plot_distances=False,
    plot_umap=False,
    size=5,
):
    # set annotations
    nearest_k = distances[1 : k + 1]
    subset_adata.obs[f"neighborhood_{center_cell}_{k}"] = np.nan
    subset_adata.obs.loc[
        subset_adata.obs.dataset == dataset, f"neighborhood_{center_cell}_{k}"
    ] = "Query"

    query_indices = subset_adata.obs[subset_adata.obs.dataset == dataset].index
    query_nearest_k = [cell for cell in nearest_k.index if cell in query_indices]

    subset_adata.obs.loc[
        nearest_k.index, f"neighborhood_{center_cell}_{k}"
    ] = "Included Reference"

    subset_adata.obs.loc[
        query_nearest_k, f"neighborhood_{center_cell}_{k}"
    ] = "Included Query"

    subset_adata.obs.loc[center_cell, f"neighborhood_{center_cell}_{k}"] = "Center"

    # get sample scores for k nearest distances

    grouped = (
        (
            subset_adata.obs.loc[nearest_k.index]
            .groupby("sample")
            .agg(
                {
                    f"distances_{center_cell}": "mean",
                    "core_or_extension": "first",
                    "ann_finest_level": "count",
                    "dataset": "first",
                }
            )
        )
        .dropna()
        .reset_index()
    )

    grouped = grouped[grouped["ann_finest_level"] >= 5]

    grouped["sample"] = grouped["sample"].cat.remove_unused_categories()
    grouped["dataset"] = grouped["dataset"].cat.remove_unused_categories()

    if return_grouped:
        # plot umap
        if plot_umap:
            sc.pl.umap(
                subset_adata,
                color=f"neighborhood_{center_cell}_{k}",
                size=size,
                palette={
                    "Center": "magenta",
                    "Query": "skyblue",
                    "Included Query": "red",
                    "Included Reference": "purple",
                },
                title=f"{center_cell} neighborhood at k = {k}",
            )

        return grouped

    sns.boxplot(data=grouped, x="core_or_extension", y=f"distances_{center_cell}")
    sns.stripplot(
        data=grouped,
        x="core_or_extension",
        y=f"distances_{center_cell}",
        hue="dataset",
    )

    plt.title(f"neighborhood_{center_cell}_{k} distance per sample")
    plt.legend(bbox_to_anchor=[1, 0.75], loc="center left", fontsize=15)
    plt.show()

    # plot umap
    sc.pl.umap(
        subset_adata,
        color=f"neighborhood_{center_cell}_{k}",
        size=size,
        palette={
            "Center": "magenta",
            "Query": "skyblue",
            "Included Query": "red",
            "Included Reference": "purple",
        },
        title=f"{center_cell} neighborhood at k = {k}",
    )

    if plot_distances:
        subset_adata.obs.loc[
            ~(
                (
                    subset_adata.obs[f"neighborhood_{center_cell}_{k}"]
                    == "Included Reference"
                )
                | (
                    subset_adata.obs[f"neighborhood_{center_cell}_{k}"]
                    == "Included Query"
                )
            ),
            f"distances_{center_cell}",
        ] = np.nan

        vmin = np.percentile(subset_adata.obs[f"distances_{center_cell}"].dropna(), 20)
        vmax = np.percentile(subset_adata.obs[f"distances_{center_cell}"].dropna(), 80)
        print(vmin)
        print(vmax)
        sc.pl.umap(
            subset_adata,
            color=f"distances_{center_cell}",
            size=size,
            cmap="hot",
            title=f"{center_cell} neighborhood at k = {k}",
            vmin=vmin,
            vmax=vmax,
        )

In [ ]:
def add_to_col_list(col_list, value):
    col_list.append(value)
    return col_list

In [ ]:
def combine_pvalues(pvalue_list):
    return scipy.stats.combine_pvalues(pvalue_list).pvalue

In [ ]:
def transfer_pvalues(adata, k, pvalues):
    adata.obs["pvalues_neighborhood"] = np.empty((len(adata), 0)).tolist()
    for cell in tqdm.tqdm(pvalues["Unnamed: 0"]):
        distances = cdist(adata[cell, :].X, adata.X).flatten()
        adata.obs[f"distances"] = distances
        distances = adata.obs[f"distances"].sort_values()
        nearest_k = distances[1 : k + 1]

        pvalue = pvalues.loc[pvalues["Unnamed: 0"] == cell, "pvalue_corr"]
        adata.obs.loc[nearest_k.index, "pvalues_neighborhood"] = (
            adata[nearest_k.index, :]
            .obs["pvalues_neighborhood"]
            .apply(add_to_col_list, value=pvalue.values[0])
        )
    adata.obs.drop(columns=["distances"], inplace=True)

    adata.obs["count_neighborhood"] = adata.obs["pvalues_neighborhood"].apply(len)
    adata.obs["pvalues_neighborhood_combine"] = adata.obs["pvalues_neighborhood"].apply(
        combine_pvalues
    )
    adata.obs["pvalues_neighborhood_mean"] = adata.obs["pvalues_neighborhood"].apply(
        np.mean
    )

    return adata

### Running the method on the whole healthy query + reference: comparing different k's

parameters:
- proportion of cells samples: 1/1000
- runs for standard deviation: 10

### Sheppard 2020

In [ ]:
plt.rcParams["figure.figsize"] = [15, 8]
plt.rcParams["axes.facecolor"] = "w"

In [ ]:
prop = 1000
for name, path in [
    ("Healthy", Sheppard_healthy),
    ("IPF", Sheppard_IPF),
    ("ScILD", Sheppard_ScILD),
]:
    stats = load_stats(path, k_list=k_list, prop=prop, runs=10)
    plt.plot(
        stats.k,
        stats.fraction,
        label=name,
    )
plt.legend()
plt.title(
    "Sheppard: Comparison of healthy, Sclerodermal and IPF on the whole embedding in dependance of nhood size k",
    fontsize=20,
)
plt.ylabel("fraction of significant nhoods", fontsize=20)
plt.xlabel("nhood size k", fontsize=20)
plt.show()


for name, path in [
    ("Healthy", Sheppard_healthy),
    ("IPF", Sheppard_IPF),
    ("ScILD", Sheppard_ScILD),
]:
    stats = load_stats(path, k_list=k_list, prop=prop, runs=10)
    plt.errorbar(
        x=stats.k, y=stats.fraction, yerr=stats.std_fraction, label=name, capsize=10
    )
plt.legend()
plt.title(
    "Sheppard: Comparison of healthy, Sclerodermal and IPF on the whole embedding in dependance of nhood size k",
    fontsize=20,
)
plt.ylabel("fraction of significant nhoods", fontsize=20)
plt.xlabel("nhood size k", fontsize=20)

## Sheppard: Further exploration

In [ ]:
adata_ext_full = sc.read_h5ad(path_hlca_ext)

adata_ext_h = adata_ext_full[
    (adata_ext_full.obs.core_or_extension == "core")
    | (
        (adata_ext_full.obs.dataset == "Sheppard_2020")
        & ~(adata_ext_full.obs.lung_condition == "Systemic sclerosis-associated ILD")
    )
].copy()

In [ ]:
del adata_ext_full

In [ ]:
adata_ext_h.obs.lung_condition

In [ ]:
aggregated_ks = []
for k in k_list:
    aggr = get_aggregation_per_cell_type(
        adata_ext_h, Sheppard_healthy, Sheppard_IPF, k, 1000, 10
    )
    aggr["k"] = k
    aggregated_ks.append(aggr)

In [ ]:
aggr_ks = pd.concat(aggregated_ks, ignore_index=True)

In [ ]:
aggr_ks["-log(p_value)"] = -np.log(aggr_ks["p_value"])

In [ ]:
agg = aggr_ks.groupby(["cell_label", "condition", "k"]).agg({"p_value": "median"})
agg_sign = agg[agg.p_value < 0.05].sort_values(by="p_value")
print(agg_sign)
cell_types_sign = agg_sign.reset_index().cell_label.unique()

In [ ]:
cell_types_sign

In [ ]:
sns.lineplot(
    data=aggr_ks[
        (aggr_ks.cell_label.isin(cell_types_sign)) & (aggr_ks.condition == "healthy")
    ],
    x="k",
    y="-log(p_value)",
    hue="cell_label",
)

plt.plot(k_list, [-np.log(0.05)] * len(k_list), color="white")

plt.legend(bbox_to_anchor=[1, 0.75], loc="center left", fontsize=15)
# plt.ylim(0, 50)

plt.title(
    "Sheppard_2020 Healthy: p-value aggregation over cell types filtered (by median) significant cell types",
    size=20,
)

plt.ylabel("-log(p-value)", size=20)
plt.xlabel("nhood size: k", size=20)

In [ ]:
sns.lineplot(
    data=aggr_ks[
        (aggr_ks.cell_label.isin(cell_types_sign)) & (aggr_ks.condition == "disease")
    ],
    x="k",
    y="-log(p_value)",
    hue="cell_label",
)

plt.plot(k_list, [-np.log(0.05)] * len(k_list), color="white")


plt.legend(bbox_to_anchor=[1, 0.75], loc="center left", fontsize=15)
# plt.ylim(0, 50)


plt.title(
    "Sheppard_2020 IPF: p-value aggregation over cell types filtered (by median) significant cell types",
    size=20,
)

plt.ylabel("-log(p-value)", size=20)
plt.xlabel("nhood size: k", size=20)

In [ ]:
df = []
for k in k_list:
    for i in range(10):
        h = load_pvalues(Sheppard_healthy, k, 1000, i)
        if h is None:
            continue
        for m in ["fisher", "pearson", "tippett", "stouffer", "mudholkar_george"]:
            pval = scipy.stats.combine_pvalues(h.pvalue_corr, method=m).pvalue
            df.append((k, m, pval, -np.log(pval)))
df = pd.DataFrame(data=df, columns=["k", "method", "pvalue", "-log(pval)"])

In [ ]:
sns.lineplot(data=df, x="k", y="-log(pval)", hue="method")
plt.plot(df.k.unique(), [-np.log(0.05)] * len(df.k.unique()), color="black")

In [ ]:
df = []
for k in k_list:
    for i in range(10):
        h = load_pvalues(Sheppard_IPF, k, 1000, i)
        if h is None:
            continue
        for m in ["fisher", "pearson", "tippett", "stouffer", "mudholkar_george"]:
            pval = scipy.stats.combine_pvalues(h.pvalue_corr, method=m).pvalue
            df.append((k, m, pval, -np.log(pval)))
df = pd.DataFrame(data=df, columns=["k", "method", "pvalue", "-log(pval)"])



In [ ]:
sns.lineplot(data=df, x="k", y="-log(pval)", hue="method")
plt.plot(df.k.unique(), [-np.log(0.05)] * len(df.k.unique()), color="black")

## Time for Umaps!: Visualizing the neighborhoods

In [ ]:
plt.rcParams["figure.figsize"] = [12, 11]

In [ ]:
sc.pp.neighbors(adata_ext_h, 15)

In [ ]:
sc.tl.umap(adata_ext_h)

In [ ]:
umap = adata_ext_h.obsm["X_umap"]
np.save(f"{Sheppard_healthy}/umap.npy", umap)

In [ ]:
adata_ext_h.obsm["X_umap"] = np.load(f"{Sheppard_healthy}/umap.npy")

In [ ]:
adata_ext_h.obs["query_health"] = np.nan
adata_ext_h.obs.loc[
    (adata_ext_h.obs.core_or_extension == "extension"), "query_health"
] = "Disease"
adata_ext_h.obs.loc[
    (adata_ext_h.obs.core_or_extension == "extension")
    & (adata_ext_h.obs.lung_condition == "Healthy"),
    "query_health",
] = "Healthy"

In [ ]:
sc.pl.umap(
    adata_ext_h,
    color="cell_label",
    size=5,
)

In [ ]:
plt.rcParams["figure.figsize"] = [15, 15]
cmap_ref = matplotlib.cm.get_cmap("Greys")
ref_samples = adata_ext_h.obs.loc[adata_ext_h.obs.core_or_extension == "core", :][
    "sample"
].unique()
n_ref_samples = len(ref_samples)
sample_floats_ref = np.arange(start=0, stop=n_ref_samples) / n_ref_samples
sample_hex_ref = [rgb2hex(cmap_ref(sample_fl)) for sample_fl in sample_floats_ref]
np.random.shuffle(
    sample_hex_ref
)  # randomize order so that not counfounded by dataset (sample order is by dataset)
# adata_ext_h_DC2_core.uns["sample_colors"] = sample_hex
ref_sample_to_col = {samp: hexcol for samp, hexcol in zip(ref_samples, sample_hex_ref)}

# generate color for query samples (here only for healthy, but can adapt to use for all):
cmap_q_healthy = matplotlib.cm.get_cmap("Blues")
q_healthy_samples = adata_ext_h.obs.loc[
    [
        cond == "Healthy" and c_or_e == "extension"
        for cond, c_or_e in zip(
            adata_ext_h.obs.lung_condition, adata_ext_h.obs.core_or_extension
        )
    ],
    "sample",
].unique()
n_q_healthy_samples = len(q_healthy_samples)
sample_floats_q_healthy = (
    np.arange(start=0, stop=n_q_healthy_samples) / n_q_healthy_samples
)
sample_hex_q_healthy = [
    rgb2hex(cmap_q_healthy(sample_fl)) for sample_fl in sample_floats_q_healthy
]

np.random.shuffle(
    sample_hex_q_healthy
)  # randomize order so that not counfounded by dataset (sample order is by dataset)
# adata_ext_h_DC2_core.uns["sample_colors"] = sample_hex
q_healthy_sample_to_col = {
    samp: hexcol for samp, hexcol in zip(q_healthy_samples, sample_hex_q_healthy)
}


# generate color for IPF samples (here only for healthy, but can adapt to use for all):
cmap_q_IPF = matplotlib.cm.get_cmap("Reds")
q_IPF_samples = adata_ext_h.obs.loc[
    [
        cond == "IPF" and c_or_e == "extension"
        for cond, c_or_e in zip(
            adata_ext_h.obs.lung_condition, adata_ext_h.obs.core_or_extension
        )
    ],
    "sample",
].unique()
n_q_IPF_samples = len(q_IPF_samples)

sample_floats_q_IPF = np.arange(start=0, stop=n_q_IPF_samples) / n_q_IPF_samples
sample_hex_q_IPF = [rgb2hex(cmap_q_IPF(sample_fl)) for sample_fl in sample_floats_q_IPF]

np.random.shuffle(
    sample_hex_q_IPF
)  # randomize order so that not counfounded by dataset (sample order is by dataset)
# adata_ext_h_DC2_core.uns["sample_colors"] = sample_hex
q_IPF_sample_to_col = {
    samp: hexcol for samp, hexcol in zip(q_IPF_samples, sample_hex_q_IPF)
}


# join color mappings into single dictionary
sample_to_color = ref_sample_to_col
sample_to_color.update(q_healthy_sample_to_col)
sample_to_color.update(q_IPF_sample_to_col)

samples = list(q_healthy_samples)
samples.extend(q_IPF_samples)
# plot
sc.pl.umap(
    adata_ext_h,
    color="sample",
    palette=sample_to_color,
    frameon=False,
    groups=samples,
    size=50,
)
plt.rcParams["figure.figsize"] = [12, 11]

In [ ]:
h = load_pvalues(Sheppard_healthy, 10000, 1000, 0)

In [ ]:
d = load_pvalues(Sheppard_IPF, 10000, 1000, 0)

In [ ]:
h.sort_values(by="pvalue_corr")[0:10]

In [ ]:
d.sort_values(by="pvalue_corr")[:10]

In [ ]:
adata_ext_h.obs["sampled_significant"] = np.nan

In [ ]:
adata_ext_h.obs.loc[h["Unnamed: 0"], "sampled_significant"] = "Sampled healthy"
adata_ext_h.obs.loc[
    h.loc[h["pvalue_corr"] < 0.05, "Unnamed: 0"], "sampled_significant"
] = "Significant healthy"

adata_ext_h.obs.loc[
    h.loc[h["pvalue_corr"] < 0.001, "Unnamed: 0"], "sampled_significant"
] = "Very Significant healthy"

In [ ]:
adata_ext_h.obs.loc[d["Unnamed: 0"], "sampled_significant"] = "Sampled disease"
adata_ext_h.obs.loc[
    d.loc[d["pvalue_corr"] < 0.05, "Unnamed: 0"], "sampled_significant"
] = "Significant disease"

adata_ext_h.obs.loc[
    d.loc[d["pvalue_corr"] < 0.001, "Unnamed: 0"], "sampled_significant"
] = "Very Significant disease"




In [ ]:
"""adata_ext_h.obs.loc[
    "GGGTTGCTCCGCGTTT-1-HCATisStab7659970", "sampled_significant"
] = "sampled for analysis (r)"

adata_ext_h.obs.loc[
    "CATGCCTGTCTTGATG_NML3_tsukui", "sampled_significant"
] = "sampled for analysis (q)"

"""

In [ ]:
base_size = 50

In [ ]:
size_factor = base_size / pd.concat([h, d])["nhood_weight"].mean()

In [ ]:
adata_ext_h.obs["plot_size"] = base_size / 10
adata_ext_h.obs.loc[h["Unnamed: 0"].to_numpy(), "plot_size"] = (
    size_factor * h.nhood_weight.to_numpy()
)
adata_ext_h.obs.loc[d["Unnamed: 0"].to_numpy(), "plot_size"] = (
    size_factor * d.nhood_weight.to_numpy()
)

In [ ]:
adata_ext_h.obs.loc[h["Unnamed: 0"].to_numpy(), "plot_size"].mean()

In [ ]:
adata_ext_h.obs["plot_size"].mean()

In [ ]:
sc.pl.umap(
    adata_ext_h,
    color="sampled_significant",
    palette={
        "Sampled disease": "salmon",
        "Significant disease": "darkred",
        "Sampled healthy": "cornflowerblue",
        "Significant healthy": "darkslateblue",
        "Very Significant disease": "red",
        "Very Significant healthy": "blue",
        # "sampled for analysis (q)": "magenta",
        # "sampled for analysis (r)": "lime",
    },
    size=adata_ext_h.obs["plot_size"].to_numpy(),
)
plt.show()

##  Distribution per k

In [ ]:
subset_adata = adata_ext_h[
    adata_ext_h.obs.lung_condition.isin(["Healthy", "Healthy (tumor adjacent)"])
].copy()

In [ ]:
plt.rcParams["figure.figsize"] = [6, 4]
plt.rcParams["axes.facecolor"] = "w"

In [ ]:
sample_sig = h[h.pvalue_corr < 0.05].sample(10)

In [ ]:
sample_insig = h[h.pvalue_corr >= 0.05].sample(10)

In [ ]:
adata_ext_h.obs.loc[
    h[h.pvalue_corr < 0.05]["Unnamed: 0"].to_numpy(), "core_or_extension"
].value_counts()

In [ ]:
adata_ext_h.obs.loc[
    h[h.pvalue_corr >= 0.05]["Unnamed: 0"].to_numpy(), "core_or_extension"
].value_counts()

In [ ]:
k_list = [
    500,
    1000,
    3000,
    5000,
    8000,
    10000,
    15000,
    30000,
    50000,
    100000,
    200000,
    300000,
]

In [ ]:
for samp in sample_sig.iterrows():
    cell = samp[1]["Unnamed: 0"]
    pval = samp[1]["pvalue_corr"]
    c_o_e = subset_adata[cell, :].obs["core_or_extension"].to_numpy()[0]
    print(cell)
    print(pval)
    print(c_o_e)
    print()

    distances = cdist(subset_adata[cell, :].X, subset_adata.X).flatten()
    subset_adata.obs[f"distances_{cell}"] = distances
    distances = subset_adata.obs[f"distances_{cell}"].sort_values()
    groupings = []
    for k in k_list:
        grouping = visualize_nhood(
            subset_adata, distances, cell, k, "Sheppard_2020", return_grouped=True
        )
        grouping["k"] = k
        groupings.append(grouping)
    subset_adata.obs.drop(columns=[f"distances_{cell}"], inplace=True)
    groupings = pd.concat(groupings)
    # print(groupings)
    sns.boxplot(data=groupings, x="k", y=f"distances_{cell}", hue="core_or_extension")
    plt.title(
        f"{c_o_e} cell neighborhood sample distribution at different k. p_10000={pval}"
    )
    plt.xticks(rotation=90)
    plt.show()

In [ ]:
for samp in sample_insig.iterrows():
    cell = samp[1]["Unnamed: 0"]
    pval = samp[1]["pvalue_corr"]
    c_o_e = subset_adata[cell, :].obs["core_or_extension"].to_numpy()[0]
    print(cell)
    print(pval)
    print(c_o_e)
    print()

    distances = cdist(subset_adata[cell, :].X, subset_adata.X).flatten()
    subset_adata.obs[f"distances_{cell}"] = distances
    distances = subset_adata.obs[f"distances_{cell}"].sort_values()
    groupings = []
    for k in k_list:
        grouping = visualize_nhood(
            subset_adata, distances, cell, k, "Sheppard_2020", return_grouped=True
        )
        grouping["k"] = k
        groupings.append(grouping)
    subset_adata.obs.drop(columns=[f"distances_{cell}"], inplace=True)
    groupings = pd.concat(groupings)
    # print(groupings)
    sns.boxplot(data=groupings, x="k", y=f"distances_{cell}", hue="core_or_extension")
    plt.title(
        f"{c_o_e} cell neighborhood sample distribution at different k. p_10000={pval}"
    )
    plt.xticks(rotation=90)
    plt.show()